# Silver Orders

Clean and transform orders data from bronze to silver layer with data quality checks: clean data, validate keys, and remove duplicates.

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/CommonHelper"

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/SilverLoader"

In [0]:
silver_table_name = get_table_name("orders", "silver")
bronze_table_name = get_table_name("orders", "bronze")

In [0]:
spark.read.table(bronze_table_name).display()

In [0]:
from pyspark.sql import functions as F

# --- Transform function for silver orders ---
def transform_orders(bronze_df):
    silver_df = (bronze_df
        .select(
            F.col("order_id"),
            F.col("customer_id"),
            F.col("order_date"),
            clean_text("order_status").alias("order_status"),
            F.col("total_amount"),
            clean_text("shipping_city").alias("shipping_city"),
            clean_text("shipping_state").alias("shipping_state"),
            F.col("created_at"),
            F.col("updated_at"),
            F.col("_bronze_ingestion_ts")
        )
    )
    silver_df = validate_keys(silver_df, ["order_id"])
    silver_df = remove_duplicates(silver_df, ["order_id"], order_col="updated_at")
    silver_df = drop_bronze_metadata(silver_df)
    return silver_df

result = incremental_silver_load(
    bronze_table_name=bronze_table_name,
    silver_table_name=silver_table_name,
    key_columns=["order_id"],
    transform_fn=transform_orders,
    update_condition="source.updated_at >= target.updated_at"
)

print(result)

In [0]:
display(spark.read.table(silver_table_name))